### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### ChatBot Evaluation


In [13]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
BAZARR_API_KEY = os.getenv("BAZARRLINK")

if not LANGSMITH_API_KEY:
    raise RuntimeError("Missing LANGSMITH_API_KEY. Add it to your .env file.")

if not BAZARR_API_KEY:
    raise RuntimeError("Missing BAZARRLINK. Add your API key to your .env file.")

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"


In [14]:
# Create or reuse the evaluation dataset.
from langsmith import Client

ls_client = Client()
dataset_name = "Chatbots Evaluation"

try:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"Using existing dataset: {dataset_name}")
except Exception as exc:
    # Only create the dataset if it does not already exist.
    # If this fails due to authentication/network issues, the exception will surface.
    from langsmith.utils import LangSmithNotFoundError
    if not isinstance(exc, LangSmithNotFoundError):
        raise
    dataset = ls_client.create_dataset(dataset_name=dataset_name)
    ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {"inputs": {"question": "What is LangChain?"},
             "outputs": {"answer": "A framework for building LLM applications"}},
            {"inputs": {"question": "What is LangSmith?"},
             "outputs": {"answer": "A platform for observing and evaluating LLM applications"}},
            {"inputs": {"question": "What is OpenAI?"},
             "outputs": {"answer": "A company that develops AI models and products"}},
            {"inputs": {"question": "What is Google?"},
             "outputs": {"answer": "A technology company known for search and other products"}},
            {"inputs": {"question": "What is Mistral?"},
             "outputs": {"answer": "An AI company that develops language models"}},
        ],
    )

print(f"Dataset ID: {dataset.id}")


Using existing dataset: Chatbots Evaluation
Dataset ID: c25008cb-9523-4cfc-a12c-56c4c87a6e6c


### Define Metrics (LLM as a Judge)

In [15]:
import random
import time
from openai import OpenAI, APIError
from langsmith import wrappers

client = OpenAI(
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1",
)
# Keep LangSmith tracing for supported OpenAI calls.
wrapped_client = wrappers.wrap_openai(client)

last_request_at = 0.0

def generate_content_with_retry(
    *,
    model: str,
    contents: str,
    system_instruction: str,
    temperature: float = 0,
    max_retries: int = 5,
):
    """Call an OpenAI-compatible chat-completions endpoint with retries."""
    global last_request_at
    retryable_status_codes = {429, 500, 502, 503, 504}
    min_interval = 12.0

    for attempt in range(max_retries):
        elapsed = time.monotonic() - last_request_at
        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)

        try:
            response = wrapped_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": contents},
                ],
                temperature=temperature,
            )
            last_request_at = time.monotonic()
            return response.choices[0].message.content or ""
        except APIError as error:
            last_request_at = time.monotonic()
            status_code = getattr(error, "status_code", None)
            if status_code not in retryable_status_codes or attempt == max_retries - 1:
                raise

            delay = min(2 ** attempt, 30) + random.uniform(0, 1)
            if status_code == 429:
                delay = max(delay, 30)
            print(
                f"API error (HTTP {status_code}). "
                f"Retry {attempt + 1}/{max_retries - 1} in {delay:.1f}s."
            )
            time.sleep(delay)

    raise RuntimeError("Request failed after all retries.")


In [16]:
import re

EVALUATOR_MODEL = "qwen/qwen3.7-flash"

eval_instructions = """You are a strict but fair evaluator of chatbot answers.
Compare the predicted answer with the reference answer. Judge factual correctness
and relevance to the question. The wording need not exactly match the reference.
Return exactly one word: CORRECT or INCORRECT. Do not add explanation or punctuation.
"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    question = inputs.get("question", "")
    reference = reference_outputs.get("answer", "")
    prediction = outputs.get("response", "")

    if not prediction:
        return False

    user_content = (
        f"Question:\n{question}\n\n"
        f"Reference answer:\n{reference}\n\n"
        f"Predicted answer:\n{prediction}\n\n"
        "Is the predicted answer factually correct and relevant?"
    )

    response = generate_content_with_retry(
        model=EVALUATOR_MODEL,
        contents=user_content,
        system_instruction=eval_instructions,
        temperature=0,
    )
    verdict = re.sub(r"[^A-Z]", "", response.strip().upper())

    if verdict == "CORRECT":
        return True
    if verdict == "INCORRECT":
        return False
    raise ValueError(f"Unexpected evaluator response: {response!r}")


In [17]:
# Simple length-based heuristic: output must be shorter than 2x reference length.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    prediction = str(outputs.get("response", "")).strip()
    reference = str(reference_outputs.get("answer", "")).strip()
    if not prediction or not reference:
        return False
    return len(prediction) < 2 * len(reference)


### Run evaluation

In [18]:
DEFAULT_INSTRUCTIONS = (
    "Answer the user's question accurately in one short, concise sentence."
)

def my_app(
    question: str,
    model: str = "qwen/qwen3.7-flash",
    instructions: str = DEFAULT_INSTRUCTIONS,
) -> str:
    return generate_content_with_retry(
        model=model,
        contents=question,
        system_instruction=instructions,
        temperature=0,
    )


In [19]:
# LangSmith target: inputs are dictionaries containing the dataset's "question" key.
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}


In [20]:
# Evaluate Qwen.
qwen_experiment_results = ls_client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision], # pyright: ignore[reportArgumentType]
    experiment_prefix="qwen-qwen3.7-flash-evaluation",
    max_concurrency=1,
    error_handling="ignore",
)


View the evaluation results for experiment: 'qwen-qwen3.7-flash-evaluation-2518396a' at:
https://smith.langchain.com/o/3eb47f54-a078-431a-862e-ae8747a3f295/datasets/c25008cb-9523-4cfc-a12c-56c4c87a6e6c/compare?selectedSessions=2afd4624-3c22-4986-8e03-5592e2b1aeb2




5it [01:17, 15.46s/it]


In [21]:
# Target for the second model; keep a separate function so evaluations do not overwrite each other.
def deepseek_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="deepseek/deepseek-v4-flash-0731free",
            instructions=(
                "Answer the user's question accurately in one short, concise sentence."
            ),
        )
    }


In [22]:
# Evaluate DeepSeek.
deepseek_experiment_results = ls_client.evaluate(
    deepseek_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="deepseek-v4-flash-0731free-evaluation",
    max_concurrency=1,
    error_handling="ignore",
) # pyright: ignore[reportCallIssue]


View the evaluation results for experiment: 'deepseek-v4-flash-0731free-evaluation-f8f589af' at:
https://smith.langchain.com/o/3eb47f54-a078-431a-862e-ae8747a3f295/datasets/c25008cb-9523-4cfc-a12c-56c4c87a6e6c/compare?selectedSessions=b89cff57-cd7b-4c8e-a74b-194fe282aca7




5it [01:42, 20.48s/it]


In [24]:
# Optional: inspect the experiment result objects in the notebook.
print("Qwen evaluation:", qwen_experiment_results)
print("DeepSeek evaluation:", deepseek_experiment_results)


Qwen evaluation: <ExperimentResults qwen-qwen3.7-flash-evaluation-2518396a>
DeepSeek evaluation: <ExperimentResults deepseek-v4-flash-0731free-evaluation-f8f589af>
